**Freeup system and gpu ram**

In [ ]:
import gc
import torch

def free_memory():
    gc.collect()                      # release unreachable Python objects
    if torch.cuda.is_available():
        torch.cuda.empty_cache()      # return cached GPU blocks to the driver
        torch.cuda.ipc_collect()      # clean up inter-process GPU memory handles
    gc.collect()

free_memory()



In [ ]:
!pip install transformers

## **Importing models from hugging face**

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

name = "facebook/bart-large-cnn" #model name
tokenizer = AutoTokenizer.from_pretrained(name) #Downloads only the tokenizer files.
model = AutoModelForSeq2SeqLM.from_pretrained(name) #download model weights into your computer

text = '''A cow is a domestic animal that can be seen in our homes. It is a four-legged animal that comes in various colours like
white, black or brown. It may also have large spots on its body. A cow has two small eyes and ears, a big nose, two sharp
horns and a long tail with hair at the end. Cows have strong teeth and a mouth that helps them to graze and feed on grass.
They also eat husks and grains and are usually seen lazing around in open fields. As we have understood what a cow looks like,
let us see the differences between its male and female counterparts in this essay on the cow. The male counterpart of the cow,
the bull, is mainly used by farmers to plough the agricultural land and draw carts. The young cow is called a calf, and the
young female is called a heifer. Generally, cows are seen in herds.'''

inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024)  #This turns your text into numbers
output = model.generate(**inputs, max_length=50, min_length=30, do_sample=False)
'''
This is where the summary gets written.
**inputs: hands over the numbers from the previous step.
max_length=50: the summary can’t be longer than 50 tokens.
min_length=30: it can’t be shorter than 30 tokens.
do_sample=False: don’t be random. With True, the model picks words with a bit of chance, so you’d get a different summary each
                 time. With False, you get the same, most likely answer every run
'''
summary = tokenizer.decode(output[0], skip_special_tokens=True)
'''
This turns the numbers back into words.
output[0]: the model can produce several answers at once, and we want the first (and only) one. Python counts from 0.
skip_special_tokens=True: the model adds invisible marker tokens like “start” and “end”. This hides them so you see clean text.
'''

print(summary)


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

A cow is a domestic animal that can be seen in our homes. It is a four-legged animal that comes in various colours like white, black or brown. A cow has two small eyes and ears, a big nose, two


In [ ]:
from transformers import pipeline

generator = pipeline("text-generation",)  #when no model is specified,it chooses the default model

text = """
America has changed dramatically during recent years. Not only has the number of
graduates in traditional engineering disciplines such as mechanical, civil,
electrical, chemical, and aeronautical engineering declined, but in most of
the premier American universities engineering curricula now concentrate on
and encourage largely the study of engineering science. As a result, there
are declining offerings in engineering subjects dealing with infrastructure,
the environment, and related issues, and greater concentration on high
technology subjects, largely supporting increasingly complex scientific
developments. While the latter is important, it should not be at the expense
of more traditional engineering.

Rapidly developing economies such as China and India, as well as other
industrial countries in Europe and Asia, continue to encourage and advance
the teaching of engineering. Both China and India, respectively, graduate
six and eight times as many traditional engineers as does the United States.
Other industrial countries at minimum maintain their output, while America
suffers an increasingly serious decline in the number of engineering graduates
and a lack of well-educated engineers.
"""

messages = [
    {
        "role": "user",
        "content": f"Summarize the following text in 2-3 sentences.\n\n{text}",
    }
]

out = generator(messages, max_new_tokens=120, do_sample=False)
print(out[0]["generated_text"][-1]["content"])

[transformers] No model was supplied, defaulted to HuggingFaceTB/SmolLM3-3B and revision a07cc9a.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


<think>
Okay, let's see. The user wants a summary of this text in 2-3 sentences. First, I need to understand the main points. The text talks about the decline in traditional engineering fields in the US compared to other countries. It mentions that US universities are shifting towards engineering science, focusing more on high-tech areas. This is leading to a shortage of engineers in infrastructure and environmental fields. On the other hand, countries like China and India are producing more traditional engineers. The US is losing ground in graduate numbers and skilled engineers.

I should highlight the contrast between the US and


**Text generation**

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name)

text = "Cow is a domestic animal that can be seen in"

inputs = tokenizer(text, return_tensors="pt")
output = model.generate(**inputs, max_new_tokens=40, do_sample=False, pad_token_id=tokenizer.eos_token_id,)

result=tokenizer.decode(output[0], skip_special_tokens=True)
print(" ".join(result.split()))  # removes newlines and extra spaces

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Cow is a domestic animal that can be seen in the wild, and is a member of the National Park Service.


In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-360M")
generator("In this course, we will teach you how to", max_length=30, num_return_sequences=2,)
#num_return_sequences=2 means two output will be genrated

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  724MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

[{'generated_text': 'In this course, we will teach you how to use computer science to make things happen.\n\nYou will learn the fundamentals of computer programs, how'},
 {'generated_text': 'In this course, we will teach you how to write and edit your own thesis statements. A thesis statement is the central idea or argument in your essay'}]

**Text translation(english to spanish)**

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

name = "Helsinki-NLP/opus-mt-en-es"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForSeq2SeqLM.from_pretrained(name)

text = "Hi, how are you?"

inputs = tokenizer(text, return_tensors="pt")
output = model.generate(**inputs, max_new_tokens=50, do_sample=False)

translation = tokenizer.decode(output[0], skip_special_tokens=True)
print(translation)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=50) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Hola, ¿cómo estás?


**Sentiment Analysis**

In [ ]:
#if sentence is positive give label=positive with score(chance) of being positive
#if sentence is negative give label=negative with score(chance) of being negative
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
print(classifier("I've been waiting for a HuggingFace course my whole life."))
print('*'*40)
print(classifier(["I've been waiting for a HuggingFace course my whole life.", "I hate this so much!"]))

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9598046541213989}]
****************************************
[{'label': 'POSITIVE', 'score': 0.9598046541213989}, {'label': 'NEGATIVE', 'score': 0.9994558691978455}]


**Zero-shot classification**

In [ ]:
#for the sentence give which of the candidate_labels is related how much with it
from transformers import pipeline

classifier = pipeline("zero-shot-classification")
classifier(
    "This is a course about the Transformers library",
    candidate_labels=["education", "politics", "business"],
)

[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

{'sequence': 'This is a course about the Transformers library',
 'labels': ['education', 'business', 'politics'],
 'scores': [0.844595193862915, 0.11197695881128311, 0.04342786595225334]}

**Mask filling(filling the gaps)**

In [ ]:
#fills the gaps
from transformers import pipeline

unmasker = pipeline("fill-mask")
unmasker("This course will teach you all about <mask> models.", top_k=2) #top_k argument controls how many possibilities you want to be displayed.

[transformers] No model was supplied, defaulted to distilbert/distilroberta-base and revision fb53ab8.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  331MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

[transformers] RobertaForMaskedLM LOAD REPORT from: distilbert/distilroberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[{'score': 0.19619743525981903,
  'token': 30412,
  'token_str': ' mathematical',
  'sequence': 'This course will teach you all about mathematical models.'},
 {'score': 0.04052695631980896,
  'token': 38163,
  'token_str': ' computational',
  'sequence': 'This course will teach you all about computational models.'}]

**Named entity recognition**

In [ ]:
#identify per(person),org(organization) and loc(location) from a sentence
from transformers import pipeline

ner = pipeline(
    "ner",
    model="dbmdz/bert-large-cased-finetuned-conll03-english",
    aggregation_strategy="simple",
)

ner("My name is Sylvain and I work at Hugging Face in Brooklyn.")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'entity_group': 'PER',
  'score': np.float32(0.9981694),
  'word': 'Sylvain',
  'start': 11,
  'end': 18},
 {'entity_group': 'ORG',
  'score': np.float32(0.9796019),
  'word': 'Hugging Face',
  'start': 33,
  'end': 45},
 {'entity_group': 'LOC',
  'score': np.float32(0.9932106),
  'word': 'Brooklyn',
  'start': 49,
  'end': 57}]

**Question answering from given context**

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",  # small, fine on Colab free tier
)

messages = [
    {
        "role": "user",
        "content": (
            "Answer the question using only the context.\n\n"
            "Context: My name is Sylvain and I work at Hugging Face in Brooklyn\n"
            "Question: Where do I work?"
        ),
    }
]

out = generator(messages, max_new_tokens=32)
print(out[0]["generated_text"][-1]["content"])

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Sylvain works at Hugging Face in Brooklyn.


**Image classification**

In [ ]:
from IPython.display import Image, display

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
display(Image(url=url))

In [ ]:
from transformers import pipeline

image_classifier = pipeline(
    task="image-classification", model="google/vit-base-patch16-224"
)
result = image_classifier(
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
)
print(result)

config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

[{'label': 'lynx, catamount', 'score': 0.43339911103248596}, {'label': 'cougar, puma, catamount, mountain lion, painter, panther, Felis concolor', 'score': 0.034805238246917725}, {'label': 'snow leopard, ounce, Panthera uncia', 'score': 0.03240460529923439}, {'label': 'Egyptian cat', 'score': 0.023973967880010605}, {'label': 'tiger cat', 'score': 0.022902335971593857}]


**Speech recogniion**

In [ ]:
from transformers import pipeline

transcriber = pipeline(
    task="automatic-speech-recognition", model="openai/whisper-large-v3"
)
result = transcriber(
    "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/mlk.flac"
)
print(result)

config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/2.07k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece token

{'text': ' I have a dream that one day this nation will rise up and live out the true meaning of its creed.'}


# **Transformer architecture**

In [ ]:
# model having encoder only
'''
#understand the text
#no sequential output
#used for text classification, sentiment analysis
'''


# model having decoder only
'''
#used for text generation
'''

# model having encoder + decoder
'''
can be used for translation
'''
'''

**Two-Phase Inference Process**



**Prefill Phase**

The prefill phase is like the preparation stage in cooking - it’s where all the initial ingredients are processed and made ready. This phase involves three key steps:


1.   **Tokenization**: Converting the input text into tokens (think of these as the basic building blocks the model understands)
2.   **Embedding Conversion**: Transforming these tokens into numerical representations that capture their meaning
3. **Initial Processing**: Running these embeddings through the model’s neural networks to create a rich understanding of the context





**Decode Phase**

After the prefill phase has processed the input, we move to the decode phase - this is where the actual text generation happens. The model generates one token at a time in what we call an autoregressive process (where each new token depends on all previous tokens).

The decode phase involves several key steps that happen for each new token:

1. **Attention Computation**: Looking back at all previous tokens to understand context
2. **Probability Calculation**: Determining the likelihood of each possible next token
3. **Token Selection**: Choosing the next token based on these probabilities
4. **Continuation Check**: Deciding whether to continue or stop generation

This phase is memory-intensive because the model needs to keep track of all previously generated tokens and their relationships.



**Managing Repetition: Keeping Output Fresh**

One common challenge with LLMs is their tendency to repeat themselves - much like a speaker who keeps returning to the same points. To address this, we use  types of penalties:

**Presence Penalty**: A fixed penalty applied to any token that has appeared before, regardless of how often. This helps prevent the model from reusing the same words.

**Frequency Penalty**: A scaling penalty that increases based on how often a token has been used. The more a word appears, the less likely it is to be chosen again.

**Controlling Generation Length**

We need ways to control how much text our LLM generates.

We can control generation length in several ways:

**Token Limits**: Setting minimum and maximum token counts

**Stop Sequences**: Defining specific patterns that signal the end of generation.
A stop sequence is a user-supplied string (or list of strings) that tells the generation loop to halt as soon as that text appears in the output, regardless of what the model “wanted” to do. It is a property of the inference call, not the model.

Example:
stop=["\n\n"] to cut off after the first paragraph

**End-of-Sequence Detection**: Letting the model naturally conclude its response

# **Using transformer**

In [ ]:
#converting sentences into tokens
from transformers import AutoTokenizer,AutoModel

name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModel.from_pretrained(name)
raw_inputs = [
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!",
]
inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt") #converting sentences into tokens. input_ids means token of sentences
print(inputs)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key                   | Status     |  | 
----------------------+------------+--+-
classifier.bias       | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 
pre_classifier.bias   | UNEXPECTED |  | 
pre_classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'input_ids': tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,
          2607,  2026,  2878,  2166,  1012,   102],
        [  101,  1045,  5223,  2023,  2061,  2172,   999,   102,     0,     0,
             0,     0,     0,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]])}


In [ ]:
#converting sentences into tokens
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

encoded_input = tokenizer("Hello, I'm a single sentence!")
print(encoded_input)

'''
input_ids: numerical representations of the sentence
token_type_ids: these tell the model which part of the input is sentence A and which is sentence B
attention_mask: Attention masks are tensors with the exact same shape as the input IDs tensor, filled with 0s and 1s.
                1s indicate the corresponding tokens should be attended to, and 0s indicate the corresponding tokens
                should not be attended to (i.e., they should be ignored by the attention layers of the model).
'''

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

{'input_ids': [101, 7592, 1010, 1045, 1005, 1049, 1037, 2309, 6251, 999, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


'\ninput_ids: numerical representations of the sentence\ntoken_type_ids: these tell the model which part of the input is sentence A and which is sentence B\nattention_mask: Attention masks are tensors with the exact same shape as the input IDs tensor, filled with 0s and 1s.\n                1s indicate the corresponding tokens should be attended to, and 0s indicate the corresponding tokens\n                should not be attended to (i.e., they should be ignored by the attention layers of the model).\n'

In [ ]:
#decode the tokens into sentence
tokenizer.decode(encoded_input["input_ids"])

"[CLS] hello, i ' m a single sentence! [SEP]"

In [ ]:
#Padding inputs
'''
If we ask the tokenizer to pad the inputs, it will make all sentences the same length by adding a special padding token
to the sentences that are shorter than the longest one
'''
encoded_input = tokenizer(
    ["How are you?", "I'm fine, thank you!"], padding=True, return_tensors="pt")
#here length of the 2 sentences aren't equal so padding add 0 to the shorter sentence to make it's length same to the longer one
print(encoded_input)

{'input_ids': tensor([[ 101, 2129, 2024, 2017, 1029,  102,    0,    0,    0,    0],
        [ 101, 1045, 1005, 1049, 2986, 1010, 4067, 2017,  999,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [ ]:
#suppose sentence is
sequences = ["I've been waiting for a HuggingFace course my whole life.", "So have I!"]
model_inputs = tokenizer(sequences)

# Will pad the sequences up to the maximum sequence(sentence) length
encoded_input = tokenizer(sequences, padding="longest")
print(encoded_input)
print("*"*40)

# Will pad the sequences up to the model max length
# (512 for BERT or DistilBERT)
encoded_input = tokenizer(sequences, padding="max_length")
print(encoded_input)
print("*"*40)

# Will pad the sequences up to the specified max length
encoded_input = tokenizer(sequences, padding="max_length", max_length=8)
print(encoded_input)

{'input_ids': [[101, 1045, 1005, 2310, 2042, 3403, 2005, 1037, 17662, 12172, 2607, 2026, 2878, 2166, 1012, 102], [101, 2061, 2031, 1045, 999, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}
****************************************
{'input_ids': [[101, 1045, 1005, 2310, 2042, 3403, 2005, 1037, 17662, 12172, 2607, 2026, 2878, 2166, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [ ]:
#torch.Size([Batch size, Sequence length, Hidden size])
'''
The vector output by the Transformer module is usually large. It generally has three dimensions:
Batch size: The number of sequences processed at a time (i.e.2 sentences in our example).
Sequence length: total number combintaion used to represent the sentence (16 in our example).
Hidden size: The vector dimension of each model input.
'''
outputs = model(**inputs)
print(outputs.last_hidden_state.shape)


torch.Size([2, 16, 768])


In [ ]:
#Truncating inputs
#when input sentence is too long we can remove the extras that exceed max_token limit from the right side(we can remove from left side also)
encoded_input = tokenizer(
    "This is a very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very long sentence.",
    truncation=True,max_length=8,
)
print(encoded_input["input_ids"])

[101, 2023, 2003, 1037, 2200, 2200, 2200, 102]


In [ ]:
sequences = ["I've been waiting for a HuggingFace course my whole life.", "So have I!"]

# Will truncate the sequences that are longer than the model max length
# (512 for BERT or DistilBERT)
encoded_input = tokenizer(sequences, truncation=True)
print(encoded_input)

# Will truncate the sequences that are longer than the specified max length
encoded_input = tokenizer(sequences, max_length=8, truncation=True)
print(encoded_input)

{'input_ids': [[101, 1045, 1005, 2310, 2042, 3403, 2005, 1037, 17662, 12172, 2607, 2026, 2878, 2166, 1012, 102], [101, 2061, 2031, 1045, 999, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1]]}
{'input_ids': [[101, 1045, 1005, 2310, 2042, 3403, 2005, 102], [101, 2061, 2031, 1045, 999, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1]]}


In [ ]:
#converting words into tokens, encoding it and decoding it
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

sentence = "Using a Transformer network is simple"
tokens = tokenizer.tokenize(sentence)
encoded_input = tokenizer.convert_tokens_to_ids(tokens)
decoded_input = tokenizer.decode(encoded_input)

print('tokens=',tokens)
print('encoded_input=',encoded_input)
print('decoded_input=',decoded_input)

tokens= ['using', 'a', 'transform', '##er', 'network', 'is', 'simple']
encoded_input= [2478, 1037, 10938, 2121, 2897, 2003, 3722]
decoded_input= using a transformer network is simple


In [ ]:
#Returns PyTorch tensors datatype
#model_inputs = tokenizer(sequences, padding=True, return_tensors="pt")

# Returns NumPy arrays datatype
#model_inputs = tokenizer(sequences, padding=True, return_tensors="np")

# **Fine tuning the model**

Here we are loading a dataset that contains 2 sentences in each row.we also have labels: non-equivalent(0) and equivalent(1). non-equivalent means the 2 sentences give different meaning while equilavent means they have same meaning.

**Example**:

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',

 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',

 'label': 1,

 'idx': 0}

 Our motive is the train the model according to training dataset to predict the labels and test it's accuracy in test dataset.

In [ ]:
#loading the dataset
from datasets import load_dataset


raw_datasets = load_dataset("nyu-mll/glue", "mrpc") #loads the dataset
raw_datasets #dataset is in dictiony format with training data,test data and validation data already divided

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [ ]:
#acessing the training data with index number
raw_datasets["train"][0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0}

In [ ]:
#checking all the features column in training dataset
raw_datasets["train"].features  #not_equivalent means 0 and equivalent means 1

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

In [ ]:
#perform tokenization and dynamic padding
from transformers import AutoTokenizer, DataCollatorWithPadding
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
#batched=True main advantage is It processes multiple examples at once, making tokenization much faster.
'''
here using map is more advantageous beacause:
-map give output in DatasetDict type while non-map give output in dict of lists
              no map             | with map
Keeps label	   No                |   Yes
All splits	   Manual	           |   Automatic
Memory	       Everything in RAM |   Batched, disk-backed
Padding	       Global max	       |   Dynamic per batch
Cached	       No                |   Yes
'''
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)  #perform dynamic padding
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

In [ ]:
#acessing the dataset
tokenized_datasets['train'][0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0,
 'input_ids': [101,
  7277,
  2180,
  5303,
  4806,
  1117,
  1711,
  117,
  2292,
  1119,
  1270,
  107,
  1103,
  7737,
  107,
  117,
  1104,
  9938,
  4267,
  12223,
  21811,
  1117,
  2554,
  119,
  102,
  11336,
  6732,
  3384,
  1106,
  1140,
  1112,
  1178,
  107,
  1103,
  7737,
  107,
  117,
  7277,
  2180,
  5303,
  4806,
  1117,
  1711,
  1104,
  9938,
  4267,
  12223,
  21811,
  1117,
  2554,
  119,
  102],
 'token_type_ids': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 'attention_mask': [1,
 

In [ ]:
#checking all
tokenized_datasets.column_names

{'train': ['sentence1',
  'sentence2',
  'label',
  'idx',
  'input_ids',
  'token_type_ids',
  'attention_mask'],
 'validation': ['sentence1',
  'sentence2',
  'label',
  'idx',
  'input_ids',
  'token_type_ids',
  'attention_mask'],
 'test': ['sentence1',
  'sentence2',
  'label',
  'idx',
  'input_ids',
  'token_type_ids',
  'attention_mask']}

In [ ]:
#no need to perform this because trainer will automatically remove unused columns,change label into labels and give output in torch
#removing unnecessary columns
tokenized_datasets=tokenized_datasets.remove_columns(["idx", "sentence1", "sentence2"]) #removing unnecessary columns
tokenized_datasets=tokenized_datasets.rename_column("label", "labels") #renaming label into labels
tokenized_datasets=tokenized_datasets.with_format("torch") #we want the output in torch dataype

In [ ]:
tokenized_datasets["train"]

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 3668
})

In [ ]:
!pip -q install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00


In [ ]:
#defining compute_metrics
import numpy as np
import evaluate
def compute_metrics(eval_preds):
    metric = evaluate.load("glue", "mrpc")
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
#defining TrainingArguments
from transformers import TrainingArguments

training_args = TrainingArguments("test-trainer",
                                  per_device_train_batch_size=16,
                                  per_device_eval_batch_size=16,
                                  num_train_epochs=4,
                                  learning_rate=2e-5,
                                  eval_strategy="epoch",
                                  weight_decay=0.01)

In [ ]:
#defining our model
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
#defining trainer
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
#perform the training
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.483437,0.830882,0.877005
2,No log,0.521304,0.850490,0.896082
3,0.243185,0.592358,0.848039,0.892361
4,0.243185,0.731934,0.848039,0.893836


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=920, training_loss=0.1743319096772567, metrics={'train_runtime': 289.108, 'train_samples_per_second': 50.749, 'train_steps_per_second': 3.182, 'total_flos': 571616989990080.0, 'train_loss': 0.1743319096772567, 'epoch': 4.0})

# **Training a new tokenizer from an old one**

Here we are fine tuning a tokenizer from "gpt2" to tokenize python code

In [ ]:
#load the dataset
from datasets import load_dataset

raw_datasets = load_dataset("code-search-net/code_search_net", "python") #this dataset contains all the python functions
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url'],
        num_rows: 412178
    })
    test: Dataset({
        features: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url'],
        num_rows: 22176
    })
    validation: Dataset({
        features: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url'],
        num_rows: 23107
    })
})

In [ ]:
def get_training_corpus():
    return (
        raw_datasets["train"][i : i + 1000]["whole_func_string"]
        for i in range(0, len(raw_datasets["train"]), 1000)
    )


training_corpus = get_training_corpus()

In [ ]:
#loading the tokonizer which will be fine tuned to create our new_tokonizer
from transformers import AutoTokenizer

old_tokenizer = AutoTokenizer.from_pretrained("gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
#training our new_tokenizer
new_tokenizer = old_tokenizer.train_new_from_iterator(training_corpus, 52000)

In [ ]:
#let us test the between the old_tokonizer and new_tokonizer
example = '''def add_numbers(a, b):
    """Add the two numbers `a` and `b`."""
    return a + b'''

tokens = old_tokenizer.tokenize(example)
print(tokens)
encoded_tokens = old_tokenizer(example)
print(encoded_tokens)
print('*'*40)
print('*'*40)
tokens = new_tokenizer.tokenize(example)
print(tokens)
encoded_tokens = new_tokenizer(example)
print(encoded_tokens)

['def', 'Ġadd', '_', 'n', 'umbers', '(', 'a', ',', 'Ġb', '):', 'Ċ', 'Ġ', 'Ġ', 'Ġ', 'Ġ"""', 'Add', 'Ġthe', 'Ġtwo', 'Ġnumbers', 'Ġ`', 'a', '`', 'Ġand', 'Ġ`', 'b', '`', '."', '""', 'Ċ', 'Ġ', 'Ġ', 'Ġ', 'Ġreturn', 'Ġa', 'Ġ+', 'Ġb']
{'input_ids': [4299, 751, 62, 77, 17024, 7, 64, 11, 275, 2599, 198, 220, 220, 220, 37227, 4550, 262, 734, 3146, 4600, 64, 63, 290, 4600, 65, 63, 526, 15931, 198, 220, 220, 220, 1441, 257, 1343, 275], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
****************************************
****************************************
['def', 'Ġadd', '_', 'numbers', '(', 'a', ',', 'Ġb', '):', 'ĊĠĠĠ', 'Ġ"""', 'Add', 'Ġthe', 'Ġtwo', 'Ġnumbers', 'Ġ`', 'a', '`', 'Ġand', 'Ġ`', 'b', '`."""', 'ĊĠĠĠ', 'Ġreturn', 'Ġa', 'Ġ+', 'Ġb']
{'input_ids': [338, 814, 63, 7288, 8, 65, 12, 314, 317, 275, 333, 2116, 299, 2073, 3991, 615, 65, 64, 393, 615, 66, 12000, 275, 345, 274, 425, 314], 'attention_mask': [1, 1

In [ ]:
#saving the tokenizer
#saves the tokenizer in collab
new_tokenizer.save_pretrained("python-code-tokenizer")